<a href="https://colab.research.google.com/github/dhanashree0498/enterprise-incident-resolution-assistant/blob/main/Enterprise_Incident_Resolution_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Enterprise Incident Resolution Assistant")
print("Project setup successful!")
print("Google Colab is working.")

Enterprise Incident Resolution Assistant
Project setup successful!
Google Colab is working.


In [5]:
import pandas as pd
import os
# craete a folder for project data
os.makedirs("data", exist_ok=True)

#Historical incidents records(examples)
incidents = [{
    "incident_id": "INC-001",
    "title":"API 500 Error After Deployment",
    "category": "API",
    "severity": "High",
    "symptoms": "API returns HTTP 500 errors after deployment.",
    "root_cause": "Missing database environment variable.",
    "resolution": "Checked application logs, restored the missing environment variable,redeployed the service, and verified API health checks."

},
             {
                 "incident_id": "INC-002",
         "title": "Database Connection Timeout",
        "category": "Database",
        "severity": "High",
        "symptoms": "Application requests fail due to database connection timeouts.",
        "root_cause": "Database connection pool exhaustion.",
        "resolution": "Reviewed active connections, identified connection leaks, corrected connection handling, and validated application recovery."
             },
             {
              "incident_id": "INC-003",
        "title": "High CPU Utilization",
        "category": "Infrastructure",
        "severity": "Medium",
        "symptoms": "Production server CPU usage remains above 90%.",
        "root_cause": "An inefficient background processing job.",
        "resolution": "Analyzed running processes, optimized the background job, adjusted scheduling, and monitored CPU utilization."
             },
             {
                "incident_id": "INC-004",
        "title": "Authentication Service Failure",
        "category": "Security",
        "severity": "High",
        "symptoms": "Users cannot log in and receive authentication errors.",
        "root_cause": "Expired authentication signing certificate.",
        "resolution": "Verified certificate expiration, renewed the certificate, updated the authentication service, and tested login functionality."
             },
             {
               "incident_id": "INC-005",
        "title": "Application Deployment Failure",
        "category": "Deployment",
        "severity": "Medium",
        "symptoms": "Application deployment fails during the release pipeline.",
        "root_cause": "Incorrect environment configuration in CI/CD pipeline.",
        "resolution": "Reviewed pipeline logs, corrected environment configuration, reran deployment and validated application availability."
             }
             ]
#convert incident records into table
df = pd.DataFrame(incidents)

#Save records to CSV
df.to_csv("data/incidents.csv", index=False)
print("incident dataset created successfully!")
print("Total incidents:",len(df))

display(df)


incident dataset created successfully!
Total incidents: 5


,incident_id,title,category,severity,symptoms,root_cause,resolution
0,INC-001,API 500 Error After Deployment,API,High,API returns HTTP 500 errors after deployment.,Missing database environment variable.,"Checked application logs, restored the missing..."
1,INC-002,Database Connection Timeout,Database,High,Application requests fail due to database conn...,Database connection pool exhaustion.,"Reviewed active connections, identified connec..."
2,INC-003,High CPU Utilization,Infrastructure,Medium,Production server CPU usage remains above 90%.,An inefficient background processing job.,"Analyzed running processes, optimized the back..."
3,INC-004,Authentication Service Failure,Security,High,Users cannot log in and receive authentication...,Expired authentication signing certificate.,"Verified certificate expiration, renewed the c..."
4,INC-005,Application Deployment Failure,Deployment,Medium,Application deployment fails during the releas...,Incorrect environment configuration in CI/CD p...,"Reviewed pipeline logs, corrected environment ..."


In [7]:
runbooks = {
    "api_troubleshooting.txt": """
API Incident Troubleshooting Runbook

Purpose:
Investigate HTTP 500 errors in production APIs.

Steps:
1. Check API application logs for exceptions.
2. Review recent deployment changes.
3. Verify required environment variables.
4. Test database connectivity.
5. Check API health endpoints.
6. Escalate if the root cause remains unknown.

Related Incident: INC-001
""",

    "database_troubleshooting.txt": """
Database Connection Troubleshooting Runbook

Purpose:
Investigate database connection timeout errors.

Steps:
1. Review database connection metrics.
2. Check active and idle connections.
3. Inspect connection pool configuration.
4. Identify potential connection leaks.
5. Validate database availability.
6. Escalate unresolved database incidents.

Related Incident: INC-002
""",

    "cpu_troubleshooting.txt": """
High CPU Utilization Runbook

Purpose:
Investigate high CPU usage on production servers.

Steps:
1. Review CPU monitoring dashboards.
2. Identify resource-intensive processes.
3. Check recently deployed background jobs.
4. Analyze application performance logs.
5. Apply approved performance fixes.
6. Monitor CPU utilization after remediation.

Related Incident: INC-003
"""
}

for filename, content in runbooks.items():
    with open(f"data/{filename}", "w") as file:
        file.write(content.strip())

print("Runbooks created successfully!")
print("Total runbooks:", len(runbooks))


Runbooks created successfully!
Total runbooks: 3


In [8]:
import os
print("Files inside data folder:\n")
for filename in sorted(os.listdir("data")):
  print(filename)

Files inside data folder:

api_troubleshooting.txt
cpu_troubleshooting.txt
database_troubleshooting.txt
incidents.csv


In [9]:
import shutil
from google.colab import files
shutil.make_archive("incident_data","zip","data")
files.download("incident_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
!pip -q install sentence-transformers faiss-cpu

In [8]:
from google.colab import files
import zipfile
import os

# Upload the backup ZIP file
uploaded = files.upload()

# Get the uploaded filename
zip_filename = next(iter(uploaded))

# Create the data folder
os.makedirs("data", exist_ok=True)

# Extract the backup
with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall("data")

# Check restored files
print("Dataset restored successfully!")
print("\nFiles inside data folder:")

for filename in sorted(os.listdir("data")):
    print("-", filename)

Saving incident_data.zip to incident_data.zip
Dataset restored successfully!

Files inside data folder:
- api_troubleshooting.txt
- cpu_troubleshooting.txt
- database_troubleshooting.txt
- incidents.csv


In [10]:
import pandas as pd
import os

# Step 1: Load historical incidents
df = pd.read_csv("data/incidents.csv")

documents = []

# Step 2: Add each incident
for _, row in df.iterrows():
    text = f"""
Incident ID: {row['incident_id']}
Title: {row['title']}
Category: {row['category']}
Severity: {row['severity']}
Symptoms: {row['symptoms']}
Root Cause: {row['root_cause']}
Resolution: {row['resolution']}
"""

    documents.append({
        "source": row["incident_id"],
        "text": text.strip()
    })


# Step 3: Load runbooks
# IMPORTANT: This loop is OUTSIDE the incident loop

for filename in sorted(os.listdir("data")):
    if filename.endswith(".txt"):
        with open(f"data/{filename}", "r") as file:
            text = file.read()

        documents.append({
            "source": filename,
            "text": text.strip()
        })


# Step 4: Print results
# IMPORTANT: This is OUTSIDE both loops

print("Documents loaded successfully!")
print("Total documents:", len(documents))

for doc in documents:
    print("-", doc["source"])


Documents loaded successfully!
Total documents: 8
- INC-001
- INC-002
- INC-003
- INC-004
- INC-005
- api_troubleshooting.txt
- cpu_troubleshooting.txt
- database_troubleshooting.txt


In [11]:
# Function to divide documents into smaller chunks
def chunk_text(text, chunk_size=80, overlap=20):

    words = text.split()
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# Store all document chunks
chunked_documents = []

for doc in documents:

    chunks = chunk_text(doc["text"])

    for chunk_number, chunk in enumerate(chunks):

        chunked_documents.append({
            "source": doc["source"],
            "chunk_id": chunk_number,
            "text": chunk
        })


# Display results
print("Document chunking completed!")
print("Total original documents:", len(documents))
print("Total chunks:", len(chunked_documents))

print("\nExample chunk:")
print(chunked_documents[0])


Document chunking completed!
Total original documents: 8
Total chunks: 8

Example chunk:
{'source': 'INC-001', 'chunk_id': 0, 'text': 'Incident ID: INC-001 Title: API 500 Error After Deployment Category: API Severity: High Symptoms: API returns HTTP 500 errors after deployment. Root Cause: Missing database environment variable. Resolution: Checked application logs, restored the missing environment variable,redeployed the service, and verified API health checks.'}


In [12]:
from sentence_transformers import SentenceTransformer

# Step 1: Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2: Extract text from document chunks
chunk_texts = [
    doc["text"] for doc in chunked_documents
]

# Step 3: Convert text into embeddings
embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Step 4: Display results
print("Embeddings generated successfully!")
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

print("\nExample embedding (first 10 values):")
print(embeddings[0][:10])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings generated successfully!
Number of embeddings: 8
Embedding dimensions: 384

Example embedding (first 10 values):
[ 0.0023394  -0.00519755  0.03178162 -0.06802906  0.03714813 -0.07830758
 -0.05481255 -0.00560389 -0.05218603  0.03156113]


In [17]:
import faiss
import numpy as np

# Step 1: Convert embeddings to float32
vectors = np.asarray(embeddings, dtype="float32")

# Step 2: Get embedding dimensions
dimension = vectors.shape[1]

# Step 3: Create FAISS index
index = faiss.IndexFlatIP(dimension)

# Step 4: Add embeddings to FAISS
index.add(vectors)

# Step 5: Display results
print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)
print("Vector dimensions:", index.d)


FAISS index created successfully!
Total vectors stored: 8
Vector dimensions: 384


In [15]:
%pip install -q faiss-cpu

In [16]:
import faiss
print("FAISS installed successfully!")
print("FAISS version:", faiss.__version__)


FAISS installed successfully!
FAISS version: 1.15.1


In [18]:
# Function to search historical incidents
def search_incidents(question, top_k=5):

    # Convert user question into embedding
    query_vector = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Search FAISS for similar documents
    scores, indices = index.search(
        query_vector,
        min(top_k, index.ntotal)
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx < 0:
            continue

        doc = chunked_documents[int(idx)]

        results.append({
            "source": doc["source"],
            "chunk_id": doc["chunk_id"],
            "score": float(score),
            "text": doc["text"]
        })

    return results


# Test question
question = "Why is my API returning HTTP 500 errors after deployment?"

results = search_incidents(question)

print("Question:", question)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Similarity:", round(result["score"], 3))
    print("Content:", result["text"][:250])


Question: Why is my API returning HTTP 500 errors after deployment?

Result 1
Source: INC-001
Similarity: 0.696
Content: Incident ID: INC-001 Title: API 500 Error After Deployment Category: API Severity: High Symptoms: API returns HTTP 500 errors after deployment. Root Cause: Missing database environment variable. Resolution: Checked application logs, restored the miss

Result 2
Source: api_troubleshooting.txt
Similarity: 0.595
Content: API Incident Troubleshooting Runbook Purpose: Investigate HTTP 500 errors in production APIs. Steps: 1. Check API application logs for exceptions. 2. Review recent deployment changes. 3. Verify required environment variables. 4. Test database connect

Result 3
Source: INC-005
Similarity: 0.416
Content: Incident ID: INC-005 Title: Application Deployment Failure Category: Deployment Severity: Medium Symptoms: Application deployment fails during the release pipeline. Root Cause: Incorrect environment configuration in CI/CD pipeline. Resolution: Review

Res

In [20]:
%pip install -q "huggingface_hub>=1.32.0,<2.0"

In [22]:
import huggingface_hub

print("Hugging Face Hub installed successfully!")
print("Version:", huggingface_hub.__version__)


Hugging Face Hub installed successfully!
Version: 1.33.0
